# Image Captioning — Test evaluation

Evaluate the fine-tuned **ViT-GPT2** model on the Flickr8k test split.
Imports all logic from `test.py` so metrics and decoding stay in sync with the CLI script.

## Configuration

In [ ]:
import importlib.util
import random
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn

# Import test.py without conflicting with the stdlib "test" package
_spec = importlib.util.spec_from_file_location(
    "caption_test", Path("test.py").resolve()
)
caption_test = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(caption_test)

DEFAULT_CONFIG = caption_test.DEFAULT_CONFIG
build_test_loader = caption_test.build_test_loader
build_tokenizer = caption_test.build_tokenizer
evaluate_test = caption_test.evaluate_test
load_model = caption_test.load_model
plot_samples = caption_test.plot_samples
show_samples = caption_test.show_samples

config = DEFAULT_CONFIG.copy()
config.update({
    "checkpoint": "checkpoints/best_model.pt",
    "data_dir": "data/flickr8k",
    "max_len": 30,
    "batch_size": 32,
    "num_beams": 4,
    "n_samples": 6,
    "seed": 42,
    # "save_predictions": "outputs/test_predictions.csv",  # uncomment to export CSV
})

random.seed(config["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Load model and checkpoint info

In [ ]:
tokenizer = build_tokenizer()
print(f"Vocabulary size: {len(tokenizer)}")

model, ckpt_info = load_model(config["checkpoint"], tokenizer, device)
ckpt_info

## Test set metrics (loss + CLIPScore)

In [ ]:
test_dataset, test_loader = build_test_loader(
    config["data_dir"], tokenizer, config["max_len"], config["batch_size"]
)
print(f"Test set: {len(test_dataset)} samples")

criterion = nn.CrossEntropyLoss(ignore_index=-100)
test_loss, test_clip = evaluate_test(
    model,
    test_loader,
    criterion,
    tokenizer,
    device,
    config["max_len"],
    config["num_beams"],
    save_predictions_path=config.get("save_predictions"),
)

print(f"Test loss:      {test_loss:.4f}")
print(f"Test CLIPScore: {test_clip:.2f}")

## Sample predictions (visual)

In [ ]:
plot_samples(
    model,
    test_dataset,
    tokenizer,
    device,
    n=config["n_samples"],
    max_len=config["max_len"],
    num_beams=config["num_beams"],
    seed=config["seed"],
    cols=3,
)
plt.show()

## Sample predictions (text)

In [ ]:
show_samples(
    model,
    test_dataset,
    tokenizer,
    device,
    config["n_samples"],
    config["max_len"],
    config["num_beams"],
    seed=config["seed"],
)

## Optional: single image demo


In [ ]:
from test import ViTTransform, generate_captions
from PIL import Image

demo_path = "data/flickr8k/Images/1000268201_693b08cb0e.jpg"
transform = ViTTransform()
tensor = transform(Image.open(demo_path).convert("RGB")).unsqueeze(0).to(device)
print(generate_captions(model, tensor, tokenizer, config["max_len"], config["num_beams"])[0])